[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [1]:
import torch
import torch.nn as nn

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
# ✏️ YOUR IMPLEMENTATION HERE
class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        # router + experts
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList([MLP(d_model, d_ff) for _ in range(num_experts)])
        self.d_model = d_model
        self.d_ff = d_ff
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x):
        # route tokens to top-k experts
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size * seq_len, self.d_model)
        scores = self.router(x)
        top_k_values, top_k_indices = torch.topk(scores, k = self.top_k, dim = -1)
        top_k_weights = torch.softmax(top_k_values, dim = -1)
        output = torch.zeros_like(x)
        for i in range(self.top_k):
            experts_indices = top_k_indices[:,i]
            experts_score = top_k_weights[:,i]
            experts_score = experts_score.unsqueeze(1)
            for j in range(self.num_experts):
                expert_mask = (experts_indices == j)
                output[expert_mask, :] += experts_score[expert_mask] * self.experts[j](x[expert_mask])
        output = output.view(batch_size, seq_len, self.d_model)
        return output

In [7]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (2.6ms)
  ✅ [2/4] Has router and experts (0.9ms)
  ✅ [3/4] Router logits shape (1.7ms)
  ✅ [4/4] Gradient flow (22.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (27.3ms total)
  Progress saved. Run status() to see your dashboard.

